# Grounded RAG Generation Workflow

Run the cells from top to bottom. The notebook reuses the chunks and persistent vector index built by `retrieval.ipynb`, prepares retrieval, inspects evidence, renders a grounded prompt, and generates an answer with sources.

In [2]:
import sys
from pathlib import Path

from chromadb.api.client import SharedSystemClient

# Reset process-local clients when a persistent index was removed or rebuilt.
SharedSystemClient.clear_system_cache()

project_root = Path.cwd()
if not (project_root / "src").exists():
    for parent in project_root.resolve().parents:
        if (parent / "src").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.core.config import Config
from src.core.logger import setup_logging
from src.generation.llm import LLM
from src.generation.prompts import PromptBuilder
from src.generation.rag_pipeline import RAGPipeline
from src.generation.response import RAGResponse, SourceReference
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.embedder import Embedder
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import Reranker
from src.retrieval.vector_store import VectorStore

config = Config()
setup_logging(config)
print(f"Project root: {project_root}")

2026-08-21 18:34:53,353 | INFO | src | Logging configured with level INFO


Project root: /Users/humengqing/Documents/Code/VSCode/doc-qa-agent


## 1. Load chunks produced by `src/document`

Reuse the chunks already parsed and split by the `src/document` pipeline instead of re-parsing the source PDF/DOCX files.

In [3]:
import json

chunks_path = project_root / "output/v2/chunks/all_chunks.json"
with chunks_path.open("r", encoding="utf-8") as chunks_file:
    chunks = json.load(chunks_file)
print(f"Loaded {len(chunks)} chunk(s) from {chunks_path}")
print(chunks[0]["chunk_id"])
print(chunks[0]["metadata"])

Loaded 306 chunk(s) from /Users/humengqing/Documents/Code/VSCode/doc-qa-agent/output/v2/chunks/all_chunks.json
Report_chunk_001
{'source': 'Report.docx', 'page': None, 'section_title': 'Abstract', 'chunk_type': 'text', 'chunk_index': 1}


## 2. Reuse the persistent vector index and rebuild the process-local BM25 index

`VectorStore` points at the same persistent ChromaDB collection that `retrieval.ipynb` already embedded and upserted, so it is opened here without calling `add_chunks` or re-encoding anything. BM25 has no on-disk persistence in this codebase, so its in-memory index is rebuilt from the loaded chunks (tokenization only, no embedding calls).

In [4]:
# Open the existing collection; do not re-embed or upsert (retrieval.ipynb already did).
embedder = Embedder(config)
vector_store = VectorStore(config, embedder=embedder)
print(f"Stored chunks: {vector_store.collection.count()}")

# BM25 is intentionally rebuilt because its index is not persisted.
bm25_retriever = BM25Retriever(chunks, config)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-08-21 18:34:59,091 | INFO | src.retrieval.embedder | Loaded embedding model: BAAI/bge-large-en-v1.5
2026-08-21 18:34:59,356 | INFO | src.retrieval.bm25_retriever | Built BM25 index for 306 chunk(s)


Stored chunks: 332


## 3. Assemble the pipeline

The individual components are retained as variables for inspection. `RAGPipeline` is the application-facing composition of the same components.

In [5]:
hybrid_retriever = HybridRetriever(vector_store, bm25_retriever, config)
reranker = Reranker(config)
prompt_builder = PromptBuilder(config)
llm = LLM(config)

pipeline = RAGPipeline(
    hybrid_retriever,
    reranker,
    prompt_builder,
    llm,
)
print("RAG pipeline is ready")

2026-08-21 18:34:59,404 | INFO | src.retrieval.reranker | Configured scadsai reranker: Qwen/Qwen3-Reranker-4B


RAG pipeline is ready


## 4. Inspect final evidence before calling the LLM

This step performs dense plus BM25 retrieval, RRF fusion, and Cross-Encoder reranking. It does not make a remote LLM request.

In [6]:
query = "What is the classification accuracy of ResNet26-V2?"

# RRF combines dense and keyword candidates into one ranked list.
hybrid_results = hybrid_retriever.search(query)

# Cross-Encoder jointly scores the query and each candidate for final ranking.
reranked_results = reranker.rerank(query, hybrid_results)

print(f"Hybrid candidates: {len(hybrid_results)}")
print(f"Final evidence chunks: {len(reranked_results)}")
for rank, chunk in enumerate(reranked_results, start=1):
    print(f"\n{rank}. {chunk['chunk_id']} | Rerank: {chunk['rerank_score']:.6f}")
    print(chunk["metadata"])
    print(chunk["text"][:300])

2026-08-21 18:35:00,027 | INFO | src.retrieval.hybrid_retriever | Fused 40 dense and 40 BM25 result(s) into 10 chunk(s)
2026-08-21 18:35:00,146 | INFO | src.retrieval.reranker | Reranked 10 candidate(s) through ScaDS.AI


Hybrid candidates: 10
Final evidence chunks: 5

1. HuMengqing_chunk_132 | Rerank: 0.983888
{'chunk_index': 132, 'section_title': '5 Summary', 'source': 'HuMengqing.pdf', 'chunk_type': 'text', 'page': 64}
To verify the effectiveness of the ResNet-V2 model, comparative experiments are conducted with other classical convolutional neural network models, including EfficientNet-B0 and VGG16. The experimental results show that the ResNet26-V2 model not only outperforms the other models in terms of classifi

2. HuMengqing_chunk_126 | Rerank: 0.154476
{'chunk_type': 'text', 'section_title': '4.4 Comparison with Other Convolutional Neural Networks', 'source': 'HuMengqing.pdf', 'page': 63, 'chunk_index': 126}
By comparing the results, it is found that ResNet26-V2 performs better in each metric relative to EfficientNet-B0 and VGG16. In addition, the ResNet26-V2 model designed in this study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary classification tasks, has

## 5. Render the grounded prompt

Inspect this output before generation when validating retrieval quality or modifying the Prompt template. It contains actual chunk IDs and metadata for traceability.

In [7]:
prompt = prompt_builder.build(query, reranked_results)
print(prompt)

# Role

You are a research document question-answering assistant. Answer the user's question only from the provided context.

# Rules

- Do not add information that is absent from the context.
- If the context does not contain enough information, say: "Based on the available documents, I cannot answer this question."
- Keep numerical values exactly as they appear in the context.
- End the answer with the chunk ID or IDs that support it.

# Context

[Chunk ID: HuMengqing_chunk_132]
Source: HuMengqing.pdf | Page: 64 | Section: 5 Summary
Content:
To verify the effectiveness of the ResNet-V2 model, comparative experiments are conducted with other classical convolutional neural network models, including EfficientNet-B0 and VGG16. The experimental results show that the ResNet26-V2 model not only outperforms the other models in terms of classification accuracy, but also uses significantly fewer parameters, which contributes to improved computational efficiency and reduced memory consumption. 

## 6. Generate an answer and display sources

This cell sends one request to ScaDS.AI. The source list is constructed from the reranked evidence rather than inferred from the generated answer.

In [8]:
# This request uses SCADS_API_KEY from .env. Do not print the key.
answer = llm.generate(prompt)
sources = tuple(
    SourceReference(
        chunk_id=chunk["chunk_id"],
        source=chunk["metadata"].get("source", "unknown source"),
        page=chunk["metadata"].get("page"),
        section_title=chunk["metadata"].get("section_title"),
    )
    for chunk in reranked_results
)
response = RAGResponse(answer=answer, sources=sources)

print(response.answer)
print("\nSources:")
for source in response.sources:
    location = f"page {source.page}" if source.page is not None else "page unknown"
    section = source.section_title or "section unknown"
    print(f"- {source.chunk_id} | {source.source} | {location} | {section}")

2026-08-21 18:35:02,665 | INFO | src.generation.llm | Generated answer with 173 character(s)


Based on the available documents, I cannot answer this question. HuMengqing_chunk_132, HuMengqing_chunk_126, HuMengqing_chunk_115, HuMengqing_chunk_002, HuMengqing_chunk_127

Sources:
- HuMengqing_chunk_132 | HuMengqing.pdf | page 64 | 5 Summary
- HuMengqing_chunk_126 | HuMengqing.pdf | page 63 | 4.4 Comparison with Other Convolutional Neural Networks
- HuMengqing_chunk_115 | HuMengqing.pdf | page 58 | 4.3 Visualization Analysis
- HuMengqing_chunk_002 | HuMengqing.pdf | page 4 | ABSTRACT
- HuMengqing_chunk_127 | HuMengqing.pdf | page 63 | 4.5 Trained Model Output


## Application shortcut

In an application, replace the manual inspection cells with `pipeline.answer(query)`. Do not run the shortcut immediately after the generation cell because it repeats retrieval and sends another LLM request.

In [9]:
response = pipeline.answer(query)
print(response.answer)

2026-08-21 18:35:03,267 | INFO | src.retrieval.hybrid_retriever | Fused 40 dense and 40 BM25 result(s) into 10 chunk(s)
2026-08-21 18:35:03,326 | INFO | src.retrieval.reranker | Reranked 10 candidate(s) through ScaDS.AI
2026-08-21 18:35:05,312 | INFO | src.generation.llm | Generated answer with 173 character(s)
2026-08-21 18:35:05,313 | INFO | src.generation.rag_pipeline | Answered question with 5 retrieved source(s)


Based on the available documents, I cannot answer this question. HuMengqing_chunk_132, HuMengqing_chunk_126, HuMengqing_chunk_115, HuMengqing_chunk_002, HuMengqing_chunk_127
